# Wrapping Gemma with Titans Memory

This notebook installs `rlms[titans]`, loads a small **Gemma 3 1B-IT**
model, and splices a Titans memory module into the final transformer
block via `TitansAugmentedLM`.  We then run a short generation and a
synthetic "memorise then recall" test.

In [ ]:
%pip install --quiet "torch>=2.3.0" "transformers>=4.45.0" "accelerate>=0.30.0"
import os
if not os.path.exists("/content/rlm"):
    !git clone --depth 1 https://github.com/alexzhang13/rlm /content/rlm 2>/dev/null || true
%pip install --quiet -e /content/rlm 2>/dev/null || %pip install --quiet -e .


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from rlm.memory import MemoryConfig, TitansAugmentedLM

# Gemma 3 1B-IT is a tiny chat-tuned Gemma that fits on a free Colab GPU.
MODEL = "google/gemma-3-1b-it"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

tok = AutoTokenizer.from_pretrained(MODEL)
base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dtype).to(device)
print("base loaded;", sum(p.numel() for p in base.parameters()) / 1e6, "M params")


## Build the memory-augmented LM

`TitansAugmentedLM` registers a forward hook on the last transformer
block.  At each forward pass it projects the block's hidden states into
the memory's `(key, value, query)` space, updates the memory with the
keys/values, reads at the queries, and adds the result back into the
hidden stream.

In [ ]:
cfg = MemoryConfig(
    key_dim=0,        # 0 = inherit the model hidden size
    value_dim=0,
    hidden_dim=512,
    n_layers=2,
    inner_lr=0.5,
    momentum=0.9,
    forget_rate=0.01,
    learnable_gates=True,
    chunk_size=8,
)
lm = TitansAugmentedLM(base, cfg, flavor="titans", insert_layer=-1).to(device)
print("memory params:", sum(p.numel() for p in lm.memory.parameters()) / 1e6, "M")


## Smoke-test generation

A vanilla generation request with the memory in the loop.

In [ ]:
prompt = "Q: What does the Titans memory module store?\nA:"
inputs = tok(prompt, return_tensors="pt").to(device)
lm.reset_memory(batch_size=1, device=device, dtype=dtype)
with torch.no_grad():
    out = lm.generate(**inputs, max_new_tokens=64, do_sample=False, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0], skip_special_tokens=True))


## Memorise-and-recall

Stream a long passage through the model so the memory module learns
the hidden association, then ask the model to recall it.  This is the
showcase for the parametric memory: the answer is **not** in the
context window of the final question, but it *is* in the memory's
learned weights.

In [ ]:
passage = (
    "Please remember the following important fact for later: "
    "The vault passcode for project Titans is GRIFFIN-2049. "
) * 8  # repeat so the memory has many rehearsal steps

# Stream the passage so the memory absorbs it.
lm.reset_memory(batch_size=1, device=device, dtype=dtype)
with torch.no_grad():
    enc = tok(passage, return_tensors="pt").to(device)
    _ = lm(**enc)

# Now query.  We *don't* include the passage in the query - only the memory
# is carrying the information.
query = "Question: What is the vault passcode for project Titans?\nAnswer:"
enc_q = tok(query, return_tensors="pt").to(device)
with torch.no_grad():
    out = lm.generate(**enc_q, max_new_tokens=24, do_sample=False, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0][enc_q.input_ids.shape[1]:], skip_special_tokens=True))


### Caveats

The Titans memory module is initialised randomly here, so a single
rehearsal isn't enough for it to reliably store arbitrary associations.
For real downstream use, you'd:

1. **Meta-train** the memory: keep the base LM frozen and train the
   memory module on a corpus of (long-context, query) pairs so the
   initial gates and projection layers learn good inductive biases.
2. **Mix memory-as-context with attention**: keep a small amount of the
   recent context in attention, and offload the long tail to memory.

See the next notebook (`titans_rlm_benchmark.ipynb`) for an end-to-end
benchmark of the system as an RLM backend.